# Download experiment data: 2026-04-23 → today

Pulls bumblebee detections and per-detection flight tracks for every day in the experiment window, for system 900 and system 939.

Output layout matches the existing convention used by `components.ipynb`:

```
data/flight_data/<YYYY-MM-DD>_system_<sysid>/
    detections.csv
    flight_tracks.csv
```

Per-uid track files are also cached under `cache/flight_data_<YYYY-MM-DD>_system_<sysid>/tracks/` so re-runs are cheap.

Run all cells top-to-bottom. Re-running is safe: anything already on disk is skipped unless `FORCE_REFRESH = True`.

## 1. Login + section/spots (one-time setup)

In [2]:
import sys
if "pats" not in sys.path:
    sys.path.insert(0, "pats")  # local scripts live in ./pats

import os
from datetime import datetime, timedelta, date
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

import logger
from pats_service import PatsService, PatsServiceError

logger.init_logger(logger=logger.logger)
load_dotenv()

user  = os.getenv("pats_user")
passw = os.getenv("pats_passw")

service  = PatsService(user, passw)
sections = service.download_sections()
section_id = sections[0]["id"]
spots = service.download_spots(section_id)

print(f"section_id={section_id}")
print("systems available:", [s["system_id"] for s in spots["c"]])

* INFO  - Successfully retrieved API token from server: https://pats-c.com - pats_service.69
* INFO  - Successfully sections from pats server - pats_service.170
* INFO  - Successfully retrieved spots from pats servers - pats_service.229


section_id=509
systems available: [900, 939]


## 2. Batch configuration

Edit `DATE_START` / `DATE_END` if you want to widen or narrow the window. By default it goes from the day the data-transfer experiment began (2026-04-23) up to *today*.

In [3]:
# --- Batch config ----------------------------------------------------------
"""TARGET_DATE = date(2026, 4, 13)     # adjust as needed; ignored if using JOBS list below


DATE_START = TARGET_DATE
DATE_END   = TARGET_DATE      # inclusive; today is fine, partial day = partial export
"""
DATE_START  = date(2026, 4,13)
DATE_END    = date(2026,4,22)
#DATE_END   = date.today() - timedelta(days=2)      # inclusive; today is fine, partial day = partial export


SYSTEMS    = [939]        # PATS-C system ids to pull
BUMBLEBEE_CLASS_ID = 12        # "Bombus terrestris"

FORCE_REFRESH = False          # True = ignore cache and re-download everything

# --- Paths (match components.ipynb layout) ---------------------------------
EXPORT_ROOT = Path("..") / ".." / "data" / "flight_data"
CACHE_ROOT_BASE = Path("cache")

# --- Build the (date, system) work list ------------------------------------
def daterange(d0, d1):
    n = (d1 - d0).days
    return [d0 + timedelta(days=i) for i in range(n + 1)]

JOBS = [(d, sid) for d in daterange(DATE_START, DATE_END) for sid in SYSTEMS]

print(f"Window : {DATE_START}  →  {DATE_END}  ({len(daterange(DATE_START, DATE_END))} days)")
print(f"Systems: {SYSTEMS}")
print(f"Total jobs: {len(JOBS)}  ({len(daterange(DATE_START, DATE_END))} days × {len(SYSTEMS)} systems)")
print(f"Export : {EXPORT_ROOT.resolve()}")

Window : 2026-04-13  →  2026-04-22  (10 days)
Systems: [939]
Total jobs: 10  (10 days × 1 systems)
Export : /Users/jaspe/Projects/Claude/Bumblebee-monitoring/data/flight_data


## 3. Download loop

For each (date, system):
1. Resolve the matching spot from `spots["c"]`.
2. Download detections for that spot on that day (cached).
3. Download each flight track (per-uid cache).
4. Concatenate tracks and write `detections.csv` + `flight_tracks.csv` into the export folder.

Re-runnable. Empty days are exported as empty CSVs so downstream code can still iterate them.

In [4]:
def export_for(target_date: date, system_id: int) -> dict:
    """Download + export detections and flight tracks for one (date, system).

    Returns a small summary dict so the caller can print/aggregate progress.
    """
    # --- Paths -------------------------------------------------------------
    folder_name   = f"{target_date.strftime('%Y-%m-%d')}_system_{system_id}"
    cache_root    = CACHE_ROOT_BASE / f"flight_data_{folder_name}"
    tracks_dir    = cache_root / "tracks"
    detections_csv_cache = cache_root / "detections.csv"
    export_dir    = EXPORT_ROOT / folder_name

    tracks_dir.mkdir(parents=True, exist_ok=True)
    export_dir.mkdir(parents=True, exist_ok=True)

    # --- FAST PATH: skip if both export files already exist ---------------
    # Same idea as components.ipynb's `if DETECTIONS_CSV.exists() and not
    # FORCE_REFRESH` short-circuit, but lifted to the (date, system) level so
    # fully-completed days don't re-iterate every cached uid on resume.
    detections_export = export_dir / "detections.csv"
    tracks_export     = export_dir / "flight_tracks.csv"
    if detections_export.exists() and tracks_export.exists() and not FORCE_REFRESH:
        try:
            n_det = len(pd.read_csv(detections_export))
            tdf   = pd.read_csv(tracks_export) if tracks_export.stat().st_size > 0 else pd.DataFrame()
            n_tr  = tdf["detection_uid"].nunique() if "detection_uid" in tdf.columns else 0
        except Exception:
            n_det, n_tr = -1, -1
        return {"date": target_date, "system": system_id, "status": "cached",
                "n_detections": n_det, "n_tracks": n_tr,
                "export_dir": str(export_dir.resolve())}

    # --- Resolve spot ------------------------------------------------------
    matching = [s for s in spots["c"] if s.get("system_id") == system_id]
    if not matching:
        return {"date": target_date, "system": system_id, "status": "no_spot",
                "n_detections": 0, "n_tracks": 0}
    spot = matching[0]

    # --- Day window --------------------------------------------------------
    day_start = datetime.combine(target_date, datetime.min.time()).replace(hour=0,  minute=0,  second=0)
    day_end   = datetime.combine(target_date, datetime.min.time()).replace(hour=23, minute=59, second=59)

    # --- Detections (cached) ----------------------------------------------
    if detections_csv_cache.exists() and not FORCE_REFRESH:
        detections = pd.read_csv(detections_csv_cache)
    else:
        df = service.download_c_detection_features(
            section_id         = section_id,
            row_id             = spot.get("row_id"),
            post_id            = spot.get("post_id"),
            system_id          = None,   # row+post is sufficient and matches components.ipynb
            detection_class_id = BUMBLEBEE_CLASS_ID,
            start_date         = day_start,
            end_date           = day_end,
        )
        if df.empty:
            detections = pd.DataFrame()
        else:
            df["row_id"]    = spot.get("row_id")
            df["post_id"]   = spot.get("post_id")
            df["system_id"] = system_id
            detections = df
        detections.to_csv(detections_csv_cache, index=False)

    # --- Flight tracks (per-uid cache) ------------------------------------
    tracks: dict[int, pd.DataFrame] = {}
    if not detections.empty:
        for uid in detections["uid"].tolist():
            uid = int(uid)
            track_path = tracks_dir / f"{uid}.csv"
            if track_path.exists() and not FORCE_REFRESH:
                trk = pd.read_csv(track_path)
            else:
                try:
                    trk = service.download_c_flight_track(section_id, uid)
                except Exception as e:
                    print(f"    uid={uid}: FAILED ({e})")
                    continue
                # Write whatever we got (empty marker is fine)
                trk.to_csv(track_path, index=False)
            if not trk.empty:
                tracks[uid] = trk

    # --- Write export files (match components.ipynb section 9) ------------
    detections.to_csv(export_dir / "detections.csv", index=False)

    if tracks:
        frames = []
        for uid, trk in tracks.items():
            t = trk.copy()
            t["detection_uid"] = uid
            t["system_id"]     = system_id
            frames.append(t)
        all_tracks = pd.concat(frames, ignore_index=True)
    else:
        all_tracks = pd.DataFrame()
    all_tracks.to_csv(export_dir / "flight_tracks.csv", index=False)

    return {
        "date":         target_date,
        "system":       system_id,
        "status":       "ok",
        "n_detections": len(detections),
        "n_tracks":     len(tracks),
        "export_dir":   str(export_dir.resolve()),
    }

In [5]:
# Run the batch -------------------------------------------------------------
summaries = []
for i, (d, sid) in enumerate(JOBS, start=1):
    print(f"[{i:>3}/{len(JOBS)}] {d}  system={sid}  ...")
    try:
        s = export_for(d, sid)
    except Exception as e:
        s = {"date": d, "system": sid, "status": f"error: {e}",
             "n_detections": 0, "n_tracks": 0}
        print(f"    !! {e}")
    summaries.append(s)
    print(f"    → {s['status']}: {s['n_detections']} detections, {s['n_tracks']} non-empty tracks")

summary_df = pd.DataFrame(summaries)
summary_df

* INFO  - Successfully downloaded c detection features - pats_service.545


[  1/10] 2026-04-13  system=939  ...


* INFO  - Successfully downloaded c flight track from: 987893 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1016492 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1016700 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017222 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017315 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017330 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017333 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017334 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017338 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017369 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017374 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1017376 - pats_service.619
* INFO  - Success

    → ok: 927 detections, 927 non-empty tracks
[  2/10] 2026-04-14  system=939  ...


* INFO  - Successfully downloaded c detection features - pats_service.545
* INFO  - Successfully downloaded c flight track from: 1030456 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1030460 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031544 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031547 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031578 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031849 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031862 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031874 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031892 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031895 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1031912 - pats_service.619
* INFO  - Successfully d

...

* INFO  - Successfully downloaded c flight track from: 1323602 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1358626 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1358984 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359000 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359001 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359003 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359004 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359008 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359063 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359071 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359375 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1359403 - pats_service.619
* INFO  - Succes

    → ok: 586 detections, 586 non-empty tracks
[  8/10] 2026-04-20  system=939  ...


* INFO  - Successfully downloaded c flight track from: 1404149 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404154 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404162 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404172 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404173 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404183 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404189 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404197 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404214 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404215 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404230 - pats_service.619
* INFO  - Successfully downloaded c flight track from: 1404231 - pats_service.619
* INFO  - Succes

    → ok: 655 detections, 655 non-empty tracks
[  9/10] 2026-04-21  system=939  ...
    → cached: 603 detections, 603 non-empty tracks
[ 10/10] 2026-04-22  system=939  ...
    → cached: 659 detections, 659 non-empty tracks


,date,system,status,n_detections,n_tracks,export_dir
0,2026-04-13,939,ok,927,927,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
1,2026-04-14,939,ok,660,660,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
2,2026-04-15,939,ok,753,753,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
3,2026-04-16,939,ok,436,436,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
4,2026-04-17,939,ok,450,450,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
5,2026-04-18,939,ok,1021,1021,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
6,2026-04-19,939,ok,586,586,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
7,2026-04-20,939,ok,655,655,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
8,2026-04-21,939,cached,603,603,/Users/jaspe/Projects/Claude/Bumblebee-monitor...
9,2026-04-22,939,cached,659,659,/Users/jaspe/Projects/Claude/Bumblebee-monitor...


## 4. Quick sanity check

Daily totals across both systems. If the experiment had real effects, this is the table the indicators in the next notebook will be built on.

In [5]:
if not summary_df.empty:
    pivot = (summary_df
             .pivot_table(index="date", columns="system",
                          values="n_detections", aggfunc="sum", fill_value=0)
             .sort_index())
    pivot["total"] = pivot.sum(axis=1)
    print(pivot.to_string())
else:
    print("No jobs ran.")

system       900  939  total
date                        
2026-04-23   793  643   1436
2026-04-24   898  534   1432
2026-04-25   842  422   1264
2026-04-26   948  541   1489
2026-04-27  1568  630   2198
2026-04-28   957  505   1462
2026-04-29  1020  452   1472
2026-04-30  1166  417   1583
2026-05-01   944  466   1410
2026-05-02  1266  992   2258
2026-05-03  1142  620   1762
2026-05-04  1329  530   1859
2026-05-05   870  787   1657
2026-05-06     8    0      8


## 5. Wipe + refetch a specific date

If a download came back partial (e.g. only a handful of tracks on a day you'd expect to be busy, like 2026-04-23 with `n_tracks = 10` for system 900 and `14` for system 939), use this cell to delete the cache **and** the export folder for one or more `(date, system)` pairs, then re-run section 3 (Download loop) above.

The fast-path skip in `export_for` checks whether both the cache files and the export files already exist - if either is gone, it re-downloads from PATS.


In [6]:
import shutil

## Check

# --- Edit these two lists, then run the cell ----------------------------
DATES_TO_REFETCH   = [date(2026, 5, 27)]   # add more dates as needed
SYSTEMS_TO_REFETCH = SYSTEMS               # default = both cameras from the config
# ------------------------------------------------------------------------

wiped = 0
for d in DATES_TO_REFETCH:
    for sid in SYSTEMS_TO_REFETCH:
        folder_name = f"{d.strftime('%Y-%m-%d')}_system_{sid}"
        cache_dir   = CACHE_ROOT_BASE / f"flight_data_{folder_name}"
        export_dir  = EXPORT_ROOT     / folder_name

        for p in (cache_dir, export_dir):
            if p.exists():
                shutil.rmtree(p)
                print(f"  wiped  {p}")
                wiped += 1
            else:
                print(f"  clean  {p}")

print(f"\nDone. Wiped {wiped} folder(s).")
print("Now re-run section 3 (Download loop) to re-fetch from PATS.")


  wiped  cache/flight_data_2026-05-27_system_900
  wiped  ../../../data/flight_data/2026-05-27_system_900
  wiped  cache/flight_data_2026-05-27_system_939
  wiped  ../../../data/flight_data/2026-05-27_system_939

Done. Wiped 4 folder(s).
Now re-run section 3 (Download loop) to re-fetch from PATS.
